In [2]:
import json
def load_sid_split(in_json):
    """
    Load train/test subject IDs from a JSON.
    """
    with open(in_json, "r", encoding="utf-8") as f:
        obj = json.load(f)
    train = set(obj["train_sids"])
    test  = set(obj["test_sids"])
    if train & test:
        raise ValueError("Loaded split has overlapping sids.")
    return train, test, obj.get("meta", {})

# --- APPLY TO ANY DATAFRAME ---

def apply_sid_split(data, train_sids, test_sids, sid_col="sid", sex_col="sex"):
    """
    Given a DataFrame and saved subject IDs, return aligned splits for all/male/female.
    """
    sid_as_str = data[sid_col].astype(str)
    is_train = sid_as_str.isin(train_sids)
    is_test  = sid_as_str.isin(test_sids)

    train_all = data[is_train].copy()
    test_all  = data[is_test].copy()

    male   = data[data[sex_col] == "M"]
    female = data[data[sex_col] == "F"]

    train_m = male[male[sid_col].astype(str).isin(train_sids)].copy()
    test_m  = male[male[sid_col].astype(str).isin(test_sids)].copy()
    train_f = female[female[sid_col].astype(str).isin(train_sids)].copy()
    test_f  = female[female[sid_col].astype(str).isin(test_sids)].copy()

    return {"all": (train_all, test_all),
            "male": (train_m, test_m),
            "female": (train_f, test_f)}

In [4]:
# =========================
# 0) Imports & Utilities
# =========================
import numpy as np
import pandas as pd
from collections import OrderedDict
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
import os
import json

os.makedirs("./preds", exist_ok=True)

# ---- metrics ----
def mae2d(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    return float(np.mean(np.linalg.norm(y_true - y_pred, axis=1)))

def pct_within_thresh(y_true, y_pred, thresh=2.0):
    d = np.linalg.norm(np.asarray(y_true) - np.asarray(y_pred), axis=1)
    return float(np.mean(d <= thresh))

# ---- sex-adaptive regressor ----
class SexAdaptiveJointXYRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, degree=2, alpha=1.0, include_bias=False, sex_col="sex", random_state=0):
        self.degree = degree; self.alpha = alpha
        self.include_bias = include_bias; self.sex_col = sex_col
        self.random_state = random_state
    def _norm_sex(self, s):
        s = str(s).strip().lower()
        if s in ["f","female"]: return "F"
        if s in ["m","male"]:   return "M"
        return "BOTH"
    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        if self.sex_col not in X.columns:
            raise ValueError(f"X must contain '{self.sex_col}'")
        sex = X[self.sex_col].map(self._norm_sex).values
        Xn  = X.drop(columns=[self.sex_col])
        self._poly = PolynomialFeatures(self.degree, include_bias=self.include_bias)
        self._scaler = StandardScaler()
        Z = self._scaler.fit_transform(self._poly.fit_transform(Xn))
        y = np.asarray(y)
        if y.ndim != 2 or y.shape[1] != 2:
            raise ValueError("y must have shape (N,2)")
        self._pooled = Ridge(alpha=self.alpha, random_state=self.random_state).fit(Z, y)
        self._heads_ = {}
        for key in ["F","M"]:
            mask = (sex == key)
            if np.any(mask):
                self._heads_[key] = Ridge(alpha=self.alpha, random_state=self.random_state).fit(Z[mask], y[mask])
        return self
    def predict(self, X):
        X = pd.DataFrame(X).copy()
        sex = X[self.sex_col].map(self._norm_sex).values
        Xn  = X.drop(columns=[self.sex_col])
        Z = self._scaler.transform(self._poly.transform(Xn))
        out = np.empty((len(X), 2), dtype=float)
        for i in range(len(X)):
            head = self._heads_.get(sex[i], self._pooled)
            out[i] = head.predict(Z[i:i+1])[0]
        return out

# ---- build/ensure pairs ----
def _norm_sex_val(val):
    if pd.isna(val): return "BOTH"
    s = str(val).strip().lower()
    if s in ("f","female"): return "F"
    if s in ("m","male"):   return "M"
    return "BOTH"

def build_xy_forecasting_dataset(df_long, hops=(1,2),
    sid_col="sid", landmark_col="landmark", sex_col="sex",
    age_col="age", x_col="x", y_col="y", drop_nonpos_dt=True):
    use = df_long[[sid_col, landmark_col, sex_col, age_col, x_col, y_col]].dropna().copy()
    use[sex_col] = use[sex_col].map(_norm_sex_val).astype("category")
    use[age_col] = pd.to_numeric(use[age_col], errors="coerce")
    use = use.dropna(subset=[age_col])
    rows = []
    for (sid, lm), g in use.groupby([sid_col, landmark_col], sort=False):
        g = g.sort_values(age_col)
        ages = g[age_col].to_numpy(float); xs = g[x_col].to_numpy(float); ys = g[y_col].to_numpy(float)
        sex = g[sex_col].iloc[0]; n = len(g)
        if n < 2: continue
        for hop in hops:
            if hop < 1: continue
            for i in range(0, n - hop):
                j = i + hop; dt = ages[j] - ages[i]
                if drop_nonpos_dt and not (dt > 0): continue
                rows.append({
                    "sid": sid, "landmark": lm, "sex": sex,
                    "age": float(ages[i]), "age_next": float(ages[j]), "dt": float(dt),
                    "x_t": float(xs[i]), "y_t": float(ys[i]),
                    "x_next": float(xs[j]), "y_next": float(ys[j]),
                    "hop": int(hop),
                })
    out = pd.DataFrame(rows)
    if not out.empty:
        out["landmark"] = out["landmark"].astype("category")
        out["sex"] = out["sex"].astype("category")
        out["hop"] = out["hop"].astype(int)
    return out

def ensure_pairs(df, hops=(1,2)):
    need = {"x_t","y_t","age","dt","sex","x_next","y_next","hop","sid","landmark"}
    return df.copy() if need.issubset(set(df.columns)) else build_xy_forecasting_dataset(df, hops=hops)

def make_design(df_pairs, landmark, hop):
    df = df_pairs[(df_pairs["landmark"] == landmark) & (df_pairs["hop"] == hop)].copy()
    X = df[["x_t","y_t","age","dt","sex"]]
    y = df[["x_next","y_next"]].to_numpy(float)
    return df, X, y





In [ ]:
# =========================
# 6) DRIVER
# =========================
# data = pd.read_csv("/data/all_landmark_series_long.csv")
data = pd.read_csv("/data/all_landmark_series_long_with_gonion.csv")
data = data[(data["age"] >= 10) & (data["age"] <= 20)].copy()
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]
train_pairs = ensure_pairs(train_all, hops=(1,2))
test_pairs  = ensure_pairs(test_all,  hops=(1,2))

# Common params
ALPHA = 1.0
THRESH_MM = 2.0
HOPS = (1)         # use (1,) if you only want step-ahead
DEGREES = ( 2)      # compare deg-1 vs deg-2

# Landmarks present in BOTH
# landmarks = sorted(set(train_pairs["landmark"]).intersection(set(test_pairs["landmark"])))
#landmarks = ['sella','nasion','porion','orbitale','u i apex','point a','u i edge','l i edge','point b','l i apex','pogonion','menton','u 6 apex','u 6 cusp','l 6 cusp','l 6 apex','gonion l','gonion u','condyle','pns','basion','u_6_mcp','l_6_mcp','ans','articular','mid gonion']
landmarks = ['gonion']

rows = []

for hop in HOPS:
    for deg in DEGREES:
        for lm in landmarks:
            df_tr, X_tr, y_tr = make_design(train_pairs, lm, hop)
            df_te, X_te, y_te = make_design(test_pairs,  lm, hop)
            if len(df_te) == 0 or len(df_tr) == 0:
                continue
            model = SexAdaptiveJointXYRegressor(degree=deg, alpha=ALPHA, sex_col="sex", random_state=0)
            model.fit(X_tr, y_tr)
            yhat = model.predict(X_te)

            rec = OrderedDict(
                landmark=str(lm), hop=int(hop), degree=int(deg),
                model="sex_adaptive",
                n_pairs_train=len(df_tr), n_pairs_test=len(df_te),
                test_MAE_2D=round(mae2d(y_te, yhat), 4),
                pct_le_2mm=round(pct_within_thresh(y_te, yhat, THRESH_MM), 4),
            )
            # sex-wise breakdown (if present)
            for key, label in [("F","female"), ("M","male")]:
                m = (X_te["sex"].astype(str).str.upper() == key).values
                if np.any(m):
                    rec[f"test_MAE_2D_{label}"] = round(mae2d(y_te[m], yhat[m]), 4)
                    rec[f"pct_le_2mm_{label}"] = round(pct_within_thresh(y_te[m], yhat[m], THRESH_MM), 4)
            rows.append(rec)

            # save preds for this landmark/deg/hop
            outp = df_te[["sid","landmark","sex","age","age_next","dt"]].copy()
            outp["x_pred"] = yhat[:,0]; outp["y_pred"] = yhat[:,1]
            outp["x_true"] = y_te[:,0]; outp["y_true"] = y_te[:,1]
            outp.to_csv(f"./preds/preds_{lm}_hop{hop}_deg{deg}_sex_adaptive.csv", index=False)

summary_sa = pd.DataFrame(rows)
summary_sa.to_csv("results_sex_adaptive_only.csv", index=False)
print("Saved: results_sex_adaptive_only.csv")
# print(summary_sa.head())

Saved: results_sex_adaptive_only.csv


save as well

In [ ]:
# === A) TRAIN + SAVE (sex_adaptive) ==========================================
# Adds model saving into your existing training loop.
# Files will be saved as: ./models_sa/{landmark}__hop{H}__deg{D}__sex_adaptive.pkl

import os, json
from collections import OrderedDict
import joblib
import numpy as np
import pandas as pd

os.makedirs("./models_sa", exist_ok=True)
os.makedirs("./preds", exist_ok=True)

def _save_sa_model(path, model, feature_cols, meta: dict):
    """
    Persist model + exact feature column order + light metadata in a single joblib.
    """
    payload = {
        "model": model,
        "feature_cols": list(feature_cols),
        "meta": dict(meta),
    }
    joblib.dump(payload, path)

# =========================
# 6) DRIVER
# =========================
# data = pd.read_csv("/data/all_landmark_series_long.csv")
data = pd.read_csv("/data/all_landmark_series_long_with_gonion.csv")
data = data[(data["age"] >= 10) & (data["age"] <= 20)].copy()
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]
train_pairs = ensure_pairs(train_all, hops=(1,2))
test_pairs  = ensure_pairs(test_all,  hops=(1,2))

# Common params
ALPHA = 1.0
THRESH_MM = 2.0
HOPS = (2,)         # use (1,) if you only want step-ahead
DEGREES = (2,)      # compare deg-1 vs deg-2

# Landmarks present in BOTH
#landmarks = ['sella','nasion','porion','orbitale','u i apex','point a','u i edge','l i edge','point b','l i apex','pogonion','menton','u 6 apex','u 6 cusp','l 6 cusp','l 6 apex','gonion l','gonion u','condyle','pns','basion','u_6_mcp','l_6_mcp','ans','articular','mid gonion']
landmarks = ['gonion']
rows = []

for hop in HOPS:
    for deg in DEGREES:
        for lm in landmarks:
            df_tr, X_tr, y_tr = make_design(train_pairs, lm, hop)
            df_te, X_te, y_te = make_design(test_pairs,  lm, hop)
            if len(df_te) == 0 or len(df_tr) == 0:
                continue

            model = SexAdaptiveJointXYRegressor(degree=deg, alpha=ALPHA, sex_col="sex", random_state=0)
            model.fit(X_tr, y_tr)
            yhat = model.predict(X_te)

            # --- save the trained model + feature order ---
            save_path = f"./models_sa/{lm.replace(' ','_')}__hop{hop}__deg{deg}__sex_adaptive.pkl"
            _save_sa_model(
                save_path, 
                model, 
                feature_cols=X_tr.columns, 
                meta={"landmark": lm, "hop": hop, "degree": deg, "model": "sex_adaptive"}
            )

            rec = OrderedDict(
                landmark=str(lm), hop=int(hop), degree=int(deg),
                model="sex_adaptive",
                n_pairs_train=len(df_tr), n_pairs_test=len(df_te),
                test_MAE_2D=round(mae2d(y_te, yhat), 4),
                pct_le_2mm=round(pct_within_thresh(y_te, yhat, THRESH_MM), 4),
                model_file=os.path.basename(save_path),
            )
            # sex-wise breakdown (if present)
            for key, label in [("F","female"), ("M","male")]:
                m = (X_te["sex"].astype(str).str.upper() == key).values
                if np.any(m):
                    rec[f"test_MAE_2D_{label}"] = round(mae2d(y_te[m], yhat[m]), 4)
                    rec[f"pct_le_2mm_{label}"] = round(pct_within_thresh(y_te[m], yhat[m], THRESH_MM), 4)
            rows.append(rec)

            # save preds for this landmark/deg/hop
            outp = df_te[["sid","landmark","sex","age","age_next","dt"]].copy()
            outp["x_pred"] = yhat[:,0]; outp["y_pred"] = yhat[:,1]
            outp["x_true"] = y_te[:,0]; outp["y_true"] = y_te[:,1]
            outp.to_csv(f"./preds/preds_{lm.replace(' ','_')}_hop{hop}_deg{deg}_sex_adaptive.csv", index=False)

summary_sa = pd.DataFrame(rows)
summary_sa.to_csv("gonion_sex_adaptive_only.csv", index=False)
print("Saved: results_sex_adaptive_only.csv and models in ./models_sa")


Saved: results_sex_adaptive_only.csv and models in ./models_sa


In [4]:
import pandas as pd
import numpy as np
from collections import OrderedDict

# --------- INPUTS (edit these) ----------
summary_path = "results_sex_adaptive_only.csv"  # your final report
female_ref_path = "report_single/female_x.csv"                # your female test file (must have a 'landmark' column)

# --------- OUTPUTS ----------
out_both   = "wide_both_by_landmark_aligned.csv"
out_female = "wide_female_by_landmark_aligned.csv"
out_male   = "wide_male_by_landmark_aligned.csv"

# --------- Load summary + female reference order ----------
df = pd.read_csv(summary_path)
if "model" in df.columns:
    df = df[df["model"].astype(str).str.lower().eq("sex_adaptive")].copy()

for c in ("hop","degree"):
    if c in df.columns: df[c] = df[c].astype(int)

ref = pd.read_csv(female_ref_path)
if "landmark" not in ref.columns:
    raise ValueError(f"'{female_ref_path}' must contain a 'landmark' column.")
ref_order = [str(x) for x in ref["landmark"].dropna().tolist()]

def make_wide(df_in: pd.DataFrame, metric_cols: list[str], metric_alias: dict[str,str]) -> pd.DataFrame:
    metric_cols = [c for c in metric_cols if c in df_in.columns]
    if not metric_cols:
        return pd.DataFrame()

    m = df_in[["landmark","hop","degree"] + metric_cols].copy().melt(
        id_vars=["landmark","hop","degree"],
        value_vars=metric_cols,
        var_name="metric", value_name="val"
    )
    m["metric"] = m["metric"].map(lambda x: metric_alias.get(x, x))
    wide = m.pivot_table(index="landmark", columns=["metric","hop","degree"], values="val", aggfunc="mean")

    # flatten columns -> metric_h{hop}_d{deg}
    wide.columns = [f"{met}_h{h}_d{d}" for (met,h,d) in wide.columns]
    wide.index = wide.index.astype(str)

    # ---- align rows to female reference order ----
    order_known = [lm for lm in ref_order if lm in wide.index]
    others = [lm for lm in wide.index if lm not in ref_order]
    wide = wide.loc[order_known + others]

    # sort columns nicely
    def sort_key(c):
        try:
            met, h, d = c.split("_")
            return (met, int(h[1:]), int(d[1:]))
        except Exception:
            return (c, 999, 999)
    wide = wide.reindex(sorted(wide.columns, key=sort_key), axis=1)

    # round
    wide = wide.applymap(lambda v: round(v, 4) if pd.notna(v) and isinstance(v, (int,float,np.floating)) else v)
    return wide.reset_index()

# ----- build the three aligned wide tables -----
wide_both   = make_wide(df, ["test_MAE_2D","pct_le_2mm"], {"test_MAE_2D":"mae2d", "pct_le_2mm":"pct2mm"})
wide_female = make_wide(df, ["test_MAE_2D_female","pct_le_2mm_female"], {"test_MAE_2D_female":"mae2d_f","pct_le_2mm_female":"pct2mm_f"})
wide_male   = make_wide(df, ["test_MAE_2D_male","pct_le_2mm_male"], {"test_MAE_2D_male":"mae2d_m","pct_le_2mm_male":"pct2mm_m"})

if not wide_both.empty:
    wide_both.to_csv(out_both, index=False)
    print(f"Saved {out_both}  shape={wide_both.shape}")
if not wide_female.empty:
    wide_female.to_csv(out_female, index=False)
    print(f"Saved {out_female}  shape={wide_female.shape}")
if not wide_male.empty:
    wide_male.to_csv(out_male, index=False)
    print(f"Saved {out_male}  shape={wide_male.shape}")


Saved wide_both_by_landmark_aligned.csv  shape=(26, 3)
Saved wide_female_by_landmark_aligned.csv  shape=(26, 3)
Saved wide_male_by_landmark_aligned.csv  shape=(26, 3)


C:\Users\alfah\AppData\Local\Temp\ipykernel_23492\1210046167.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide = wide.applymap(lambda v: round(v, 4) if pd.notna(v) and isinstance(v, (int,float,np.floating)) else v)
C:\Users\alfah\AppData\Local\Temp\ipykernel_23492\1210046167.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide = wide.applymap(lambda v: round(v, 4) if pd.notna(v) and isinstance(v, (int,float,np.floating)) else v)
C:\Users\alfah\AppData\Local\Temp\ipykernel_23492\1210046167.py:59: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  wide = wide.applymap(lambda v: round(v, 4) if pd.notna(v) and isinstance(v, (int,float,np.floating)) else v)


skip more as long as 1 m
1,2,4,8 skip years  

max and mean of avg without outlier, 2mm std is outlier

In [1]:
# === B) EVALUATE YEAR-BASED SKIPS FOR SAVED SEX-ADAPTIVE MODELS ==============

import os, re
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

def _load_sa_payload(pkl_path):
    payload = joblib.load(pkl_path)
    # expected: {"model": ..., "feature_cols": [...], "meta": {...}}
    return payload["model"], payload["feature_cols"], payload.get("meta", {})

def _extract_meta_from_sa_filename(name: str):
    """
    Expect filenames like:
      '{landmark}__hop{H}__deg{D}__sex_adaptive.pkl'
    Returns: (landmark, hop, degree)
    """
    base = Path(name).stem
    # split landmark vs rest
    if "__hop" not in base:
        # fallback: treat everything before first "__" as landmark
        parts = base.split("__")
        landmark = parts[0].replace("_", " ")
        hop = None; deg = None
        return landmark, hop, deg

    landmark_part, rest = base.split("__hop", 1)
    landmark = landmark_part.replace("_", " ")
    m = re.match(r"(\d+)__deg(\d+)__sex_adaptive", rest)
    if m:
        hop = int(m.group(1)); deg = int(m.group(2))
    else:
        hop = None; deg = None
    return landmark, hop, deg

def _build_year_skip_pairs(df_long_lm: pd.DataFrame, skip_years: int, tol: float = 0.34):
    """
    Build age-based skip pairs for a single landmark dataframe.
    Produces a tidy DF with columns:
      sid, sex, age (source), age_target, dt, x_t, y_t, x_true, y_true
    Rules:
      - for each source age 'a', search a target age within |age - (a + skip_years)| <= tol
      - if multiple, pick nearest in absolute difference
      - dt is the actual difference (age_target - age)
    """
    rows = []
    for sid, g in df_long_lm.groupby("sid"):
        gg = g.dropna(subset=["x","y","age"]).sort_values("age")
        if len(gg) < 2:
            continue
        ages = gg["age"].to_numpy(float)
        xs   = gg["x"].to_numpy(float)
        ys   = gg["y"].to_numpy(float)
        sex_vals = gg["sex"].astype(str).to_numpy()

        # We’ll need fast nearest search by age. Do a simple linear scan (series are short).
        for i, a in enumerate(ages):
            target = a + skip_years
            # find j with minimal |ages[j]-target|
            j = np.argmin(np.abs(ages - target))
            if abs(ages[j] - target) <= tol and ages[j] > a:
                rows.append({
                    "sid": sid,
                    "sex": sex_vals[i],
                    "age": float(a),
                    "age_target": float(ages[j]),
                    "dt": float(ages[j] - a),
                    "x_t": float(xs[i]),
                    "y_t": float(ys[i]),
                    "x_true": float(xs[j]),
                    "y_true": float(ys[j]),
                })
    return pd.DataFrame(rows)

def _make_X_for_sa(df_pairs: pd.DataFrame, feature_cols: list, sex_col="sex"):
    """
    Build the feature matrix the sex-adaptive model expects.
    We assume your training 'make_design(...)' created a design with *named* columns
    that include at least: x_t (or x), y_t (or y), age, dt, sex, etc., before the
    PolyFeatures/Ridge steps.

    To be robust, we create a minimally-necessary base with common names:
      - 'x_t','y_t','age','dt','sex'
    Then reindex to the exact feature_cols order (missing cols -> fill 0).
    If your original 'make_design' used 'x'/'y' instead of 'x_t'/'y_t', add both.
    """
    X_base = pd.DataFrame(index=df_pairs.index)

    # Common primitives available from df_pairs:
    X_base["x_t"] = df_pairs["x_t"].astype(float)
    X_base["y_t"] = df_pairs["y_t"].astype(float)
    X_base["age"] = df_pairs["age"].astype(float)
    X_base["dt"]  = df_pairs["dt"].astype(float)
    X_base[sex_col] = df_pairs["sex"].astype(str)

    # Also provide aliases 'x','y' (harmless if unused)
    X_base["x"] = X_base["x_t"]
    X_base["y"] = X_base["y_t"]

    # Ensure all expected feature columns exist (fill zeros for unknowns)
    for c in feature_cols:
        if c not in X_base.columns:
            # categorical dummies are usually created inside the pipeline;
            # for unknown raw columns, fill zeros
            X_base[c] = 0.0

    # Order exact feature columns
    X = X_base[feature_cols].copy()
    return X

def _mae2d_cols(df_pairs: pd.DataFrame, x_pred_col="x_pred", y_pred_col="y_pred"):
    d = np.hypot(df_pairs[x_pred_col] - df_pairs["x_true"],
                 df_pairs[y_pred_col] - df_pairs["y_true"])
    return float(np.mean(d)), d

def per_sid_and_per_landmark_2d_year_skips_sa(pkl_path: str,
                                              data: pd.DataFrame,
                                              skips=(1,2,4,8),
                                              tol_years: float = 0.34,
                                              sex_col: str = "sex"):
    """
    Evaluate a single saved sex-adaptive model over multiple *year-based* skips.
    Returns (per_sid_df, per_landmark_df) similar to your sep-xy helper.

    per_sid_df columns:
      [landmark, sex, sid, regime, mae_2d, n_pairs]

    per_landmark_df columns:
      [landmark, sex, regime, mae_2d, n_pairs]
    """
    model, feature_cols, meta = _load_sa_payload(pkl_path)
    landmark_file, hop0, deg0 = _extract_meta_from_sa_filename(Path(pkl_path).name)
    landmark = meta.get("landmark", landmark_file)
    # filter this landmark
    df_lm = data[data["landmark"] == landmark].copy()
    if df_lm.empty:
        return pd.DataFrame(), pd.DataFrame()

    out_rows_sid = []
    out_rows_lm  = []

    for k in skips:
        pairs_k = _build_year_skip_pairs(df_lm, skip_years=k, tol=tol_years)
        if pairs_k.empty:
            continue

        X_k = _make_X_for_sa(pairs_k, feature_cols, sex_col=sex_col)
        yhat = model.predict(X_k)  # shape (N, 2)

        pairs_k = pairs_k.copy()
        pairs_k["x_pred"] = yhat[:,0]
        pairs_k["y_pred"] = yhat[:,1]
        mae_k, dvec = _mae2d_cols(pairs_k)

        # per-sid
        for (sid, sex_val), g in pairs_k.groupby(["sid","sex"]):
            m, _ = _mae2d_cols(g)
            out_rows_sid.append({
                "landmark": landmark,
                "sex": str(sex_val),
                "sid": sid,
                "regime": f"skip{k}y",     # explicit YEAR-based skip
                "mae_2d": round(m, 4),
                "n_pairs": len(g),
            })

        # per-landmark per-sex aggregate
        for sex_val, g in pairs_k.groupby("sex"):
            m, _ = _mae2d_cols(g)
            out_rows_lm.append({
                "landmark": landmark,
                "sex": str(sex_val),
                "regime": f"skip{k}y",
                "mae_2d": round(m, 4),
                "n_pairs": len(g),
            })

        # also overall (sex='both')
        m_all, _ = _mae2d_cols(pairs_k)
        out_rows_lm.append({
            "landmark": landmark,
            "sex": "both",
            "regime": f"skip{k}y",
            "mae_2d": round(m_all, 4),
            "n_pairs": len(pairs_k),
        })

    per_sid = pd.DataFrame(out_rows_sid)
    per_lm  = pd.DataFrame(out_rows_lm)
    return per_sid, per_lm

def run_year_skip_mae2d_over_sa_models(models_dir: str,
                                       data_test: pd.DataFrame,
                                       skips=(1,2,4,8),
                                       tol_years: float = 0.34):
    """
    Iterate all saved sex-adaptive models in models_dir and compute year-based
    skip errors on the given TEST dataframe.
    Aggregates across landmarks.
    """
    dfs_sid, dfs_lm = [], []
    for p in sorted(glob(f"{models_dir}/*__hop1__deg2__sex_adaptive.pkl")):
        per_sid, per_lm = per_sid_and_per_landmark_2d_year_skips_sa(
            pkl_path=p,
            data=data_test,
            skips=skips,
            tol_years=tol_years
        )
        if not per_sid.empty: dfs_sid.append(per_sid)
        if not per_lm.empty:  dfs_lm.append(per_lm)

    per_sid_all = pd.concat(dfs_sid, ignore_index=True) if dfs_sid else pd.DataFrame()
    per_lm_all  = pd.concat(dfs_lm,  ignore_index=True) if dfs_lm  else pd.DataFrame()
    return per_sid_all, per_lm_all


In [ ]:
# ---- Example end-to-end with your split helpers (YEAR-based skips) ----
# df_long must contain: sid, age, x, y, sex ('M'/'F'), landmark
df_long = pd.read_csv("/data/all_landmark_series_long.csv")
split_json = "/data/splits/sid_split_v1.json"
train_sids, test_sids, _ = load_sid_split(split_json)
df_long = df_long[(df_long["age"] >= 10) & (df_long["age"] <= 20)].copy()
re_splits = apply_sid_split(df_long, train_sids, test_sids)

models_dir = "./models_sa"    # saved in Part A

for pop in ["all"]:
    _, test_df = re_splits[pop]  # only use test
    per_sid, per_lm = run_year_skip_mae2d_over_sa_models(
        models_dir=models_dir,
        data_test=test_df,
        skips=(2,4),
        tol_years=0.34,       # ~4 months window; adjust if your time grid is coarser
    )

    per_sid.to_csv(f"per_sid_mae2d_{pop}_YEAR_skips_1.csv", index=False)
    per_lm.to_csv(f"per_landmark_mae2d_{pop}_YEAR_skips_1.csv", index=False)

    # Wide convenience views (one row per landmark/sex or landmark/sex/sid)
    if not per_lm.empty:
        wide_lm = per_lm.pivot_table(index=["landmark","sex"], columns="regime", values="mae_2d").reset_index()
        wide_lm.to_csv(f"per_landmark_mae2d_{pop}_YEAR_skips_wide.csv", index=False,float_format="%.4f")
    if not per_sid.empty:
        wide_sid = per_sid.pivot_table(index=["landmark","sex","sid"], columns="regime", values="mae_2d").reset_index()
        wide_sid.to_csv(f"per_sid_mae2d_{pop}_YEAR_skips_wide.csv", index=False,float_format="%.4f")

print("Done (sex-adaptive, YEAR-based skips).")


Done (sex-adaptive, YEAR-based skips).


In [12]:
def per_sid_and_per_landmark_2d_year_skips_sa(pkl_path: str,
                                              data: pd.DataFrame,
                                              skips=(1,2,4,8),
                                              tol_years: float = 0.34,
                                              sex_col: str = "sex"):
    model, feature_cols, meta = _load_sa_payload(pkl_path)
    landmark_file, _, _ = _extract_meta_from_sa_filename(Path(pkl_path).name)
    landmark = meta.get("landmark", landmark_file)
    
    df_lm = data[data["landmark"] == landmark].copy()
    if df_lm.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    out_rows_sid = []
    out_rows_lm  = []
    all_raw_preds = [] 

    for k in skips:
        pairs_k = _build_year_skip_pairs(df_lm, skip_years=k, tol=tol_years)
        if pairs_k.empty:
            continue

        X_k = _make_X_for_sa(pairs_k, feature_cols, sex_col=sex_col)
        yhat = model.predict(X_k)

        pairs_k = pairs_k.copy()
        pairs_k["x_pred"] = yhat[:,0]
        pairs_k["y_pred"] = yhat[:,1]
        
        # --- ADDING THE SKIP LABEL HERE ---
        pairs_k["skip_label"] = f"{k}y" 
        pairs_k["landmark"] = landmark

        # Create the tidy export for this specific skip
        raw_export = pairs_k[[
            "sid", "landmark", "sex", "skip_label", "age", "age_target", "dt", 
            "x_pred", "y_pred", "x_true", "y_true"
        ]].copy()
        
        raw_export = raw_export.rename(columns={"age_target": "age_next"})
        all_raw_preds.append(raw_export)
        # ----------------------------------

        # (Existing MAE calculation logic remains the same)
        for (sid, sex_val), g in pairs_k.groupby(["sid","sex"]):
            m, _ = _mae2d_cols(g)
            out_rows_sid.append({
                "landmark": landmark, "sex": str(sex_val), "sid": sid,
                "regime": f"skip{k}y", "mae_2d": round(m, 4), "n_pairs": len(g),
            })

        for sex_val, g in pairs_k.groupby("sex"):
            m, _ = _mae2d_cols(g)
            out_rows_lm.append({
                "landmark": landmark, "sex": str(sex_val),
                "regime": f"skip{k}y", "mae_2d": round(m, 4), "n_pairs": len(g),
            })

    per_sid = pd.DataFrame(out_rows_sid)
    per_lm  = pd.DataFrame(out_rows_lm)
    raw_preds_df = pd.concat(all_raw_preds, ignore_index=True) if all_raw_preds else pd.DataFrame()
    
    return per_sid, per_lm, raw_preds_df
# Update the main runner function as well
def run_year_skip_mae2d_over_sa_models(models_dir: str, data_test: pd.DataFrame):
    dfs_sid, dfs_lm, dfs_raw = [], [], []
    
    for p in sorted(glob(f"{models_dir}/*__hop1__deg2__sex_adaptive.pkl")):
        per_sid, per_lm, raw_preds = per_sid_and_per_landmark_2d_year_skips_sa(
            pkl_path=p, data=data_test
        )
        if not per_sid.empty: dfs_sid.append(per_sid)
        if not per_lm.empty:  dfs_lm.append(per_lm)
        if not raw_preds.empty: dfs_raw.append(raw_preds)

    per_sid_all = pd.concat(dfs_sid, ignore_index=True) if dfs_sid else pd.DataFrame()
    per_lm_all  = pd.concat(dfs_lm, ignore_index=True) if dfs_lm else pd.DataFrame()
    raw_all     = pd.concat(dfs_raw, ignore_index=True) if dfs_raw else pd.DataFrame()
    
    return per_sid_all, per_lm_all, raw_all

In [ ]:
# ---- Example end-to-end with your split helpers (YEAR-based skips) ----
# df_long must contain: sid, age, x, y, sex ('M'/'F'), landmark
df_long = pd.read_csv("/data/all_landmark_series_long_with_gonion.csv")
split_json = "/data/splits/sid_split_v1.json"
train_sids, test_sids, _ = load_sid_split(split_json)
df_long = df_long[(df_long["age"] >= 10) & (df_long["age"] <= 20)].copy()
re_splits = apply_sid_split(df_long, train_sids, test_sids)

models_dir = "./models_sa"    # saved in Part A

for pop in ["all"]:
    _, test_df = re_splits[pop]  # only use test
    _, _ ,raw_all = run_year_skip_mae2d_over_sa_models(
        models_dir=models_dir,
        data_test=test_df,
    )

    raw_all.to_csv(f"raw_predictions_{pop}_YEAR_skips_2_4.csv", index=False)
    # per_sid.to_csv(f"per_sid_mae2d_{pop}_YEAR_skips_1.csv", index=False)
    # per_lm.to_csv(f"per_landmark_mae2d_{pop}_YEAR_skips_1.csv", index=False)

    # Wide convenience views (one row per landmark/sex or landmark/sex/sid)
    # if not per_lm.empty:
    #     wide_lm = per_lm.pivot_table(index=["landmark","sex"], columns="regime", values="mae_2d").reset_index()
    #     wide_lm.to_csv(f"per_landmark_mae2d_{pop}_YEAR_skips_wide.csv", index=False,float_format="%.4f")
    # if not per_sid.empty:
    #     wide_sid = per_sid.pivot_table(index=["landmark","sex","sid"], columns="regime", values="mae_2d").reset_index()
    #     wide_sid.to_csv(f"per_sid_mae2d_{pop}_YEAR_skips_wide.csv", index=False,float_format="%.4f")

print("Done (sex-adaptive, YEAR-based skips).")

Done (sex-adaptive, YEAR-based skips).


In [16]:
import pandas as pd

# Load your master file
file_path = "raw_predictions_all_YEAR_skips_2_4.csv"
df = pd.read_csv(file_path)

skips = [2, 4]
landmarks = ["point b", "pogonion", "menton", "condyle", 'gonion']

for lm in landmarks:
    for s in skips:
        # 1. Create the label string as it appears in your 'skip_label' column
        skip_str = f"{s}y"
        
        # 2. Filter the dataframe for the specific landmark and skip
        # We use .lower() to ensure the matching isn't broken by capitalization
        filtered_df = df[
            (df['landmark'].str.lower() == lm.lower()) & 
            (df['skip_label'] == skip_str)
        ]
        
        # 3. Check if we actually have data before saving
        if not filtered_df.empty:
            # Format the filename: e.g., "point_b_2.csv"
            clean_lm_name = lm.replace(" ", "_")
            output_name = f"predes_{clean_lm_name}_{s}.csv"
            
            # 4. Save the file
            filtered_df.to_csv(output_name, index=False)
            print(f"Saved: {output_name} ({len(filtered_df)} rows)")
        else:
            print(f"No data found for {lm} at skip {s}")

Saved: predes_point_b_2.csv (115 rows)
Saved: predes_point_b_4.csv (65 rows)
Saved: predes_pogonion_2.csv (111 rows)
Saved: predes_pogonion_4.csv (64 rows)
Saved: predes_menton_2.csv (113 rows)
Saved: predes_menton_4.csv (65 rows)
Saved: predes_condyle_2.csv (114 rows)
Saved: predes_condyle_4.csv (64 rows)
Saved: predes_gonion_2.csv (115 rows)
Saved: predes_gonion_4.csv (65 rows)


In [ ]:
data = pd.read_csv("/data/all_landmark_series_long.csv")
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]

In [3]:
train_all

,sid,sex,angle_class,landmark,age_idx,age,x,y,dx,dy
0,001,M,Class I,sella,1,6.92,0.0,0.0,NaN,NaN
1,001,M,Class I,sella,2,7.75,0.0,0.0,0.0,0.0
2,001,M,Class I,sella,3,8.50,0.0,0.0,0.0,0.0
3,001,M,Class I,sella,4,9.58,0.0,0.0,0.0,0.0
4,001,M,Class I,sella,5,10.67,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
66914,U95,F,Class II,articular,5,13.00,-27.6,-27.3,-5.8,-3.6
66915,U95,F,Class II,articular,6,14.00,-26.4,-26.8,-4.6,-3.1
66916,U95,F,Class II,articular,7,16.00,-27.5,-26.3,-5.6,-2.6
66917,U95,F,Class II,articular,8,17.00,NaN,NaN,NaN,NaN


In [4]:
import numpy as np
import pandas as pd

# REQUIRED columns: sid, landmark, age_idx, x, y
# Optional/existing dx, dy will be overwritten in the returned frame if you assign back.

def recompute_dxdy_with_diagnostics(df: pd.DataFrame,
                                    sid_col="sid", lm_col="landmark",
                                    agei_col="age_idx", x_col="x", y_col="y"):
    out = df.copy()

    # Basic hygiene
    out = out.replace([np.inf, -np.inf], np.nan)
    # Ensure dtypes
    for c in [agei_col, x_col, y_col]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    # Sort for stable diffs
    out = out.sort_values([sid_col, lm_col, agei_col]).reset_index(drop=True)

    # Flag duplicates per (sid, landmark, age_idx)
    dup_mask = out.duplicated(subset=[sid_col, lm_col, agei_col], keep=False)
    out["dup_age_idx"] = dup_mask

    # Within each (sid, landmark), compute next-step forward diffs
    def _per_group(g):
        g = g.sort_values(agei_col).copy()

        # Next values
        g["x_next"] = g[x_col].shift(-1)
        g["y_next"] = g[y_col].shift(-1)
        g["agei_next"] = g[agei_col].shift(-1)

        # Gap size in age_idx (should be 1 for consecutive)
        g["agei_gap"] = g["agei_next"] - g[agei_col]

        # Forward deltas t -> t+1
        g["dx_new"] = g["x_next"] - g[x_col]
        g["dy_new"] = g["y_next"] - g[y_col]

        # Build diagnostics for NaNs / invalid deltas
        reason = np.full(len(g), "", dtype=object)

        # Missing xy at current or next row
        missing_xy = g[[x_col, y_col, "x_next", "y_next"]].isna().any(axis=1)
        reason = np.where(missing_xy, "missing_xy", reason)

        # Non-consecutive age_idx (gap != 1) → invalidate
        gap_bad = (g["agei_gap"] != 1) | g["agei_gap"].isna()
        reason = np.where((reason == "") & gap_bad, "non_consecutive_age_idx", reason)

        # Last row in the sequence (no next)
        last_row = g["agei_next"].isna()
        reason = np.where((reason == "") & last_row, "last_in_sequence", reason)

        # Duplicated age_idx in this group
        if g[agei_col].duplicated(keep=False).any():
            # mark all dup rows; you may want to aggregate/average instead
            dup_local = g[agei_col].duplicated(keep=False)
            reason = np.where((reason == "") & dup_local, "duplicate_age_idx", reason)

        # If any reason (non-empty), we nullify dx_new/dy_new
        mask_invalid = reason != ""
        g.loc[mask_invalid, ["dx_new", "dy_new"]] = np.nan
        g["dxdy_reason"] = reason

        return g

    out = out.groupby([sid_col, lm_col], group_keys=False).apply(_per_group)

    # Summary diagnostics
    total = len(out)
    nan_rows = out[["dx_new", "dy_new"]].isna().any(axis=1).sum()
    print(f"[dx,dy] rows with NaN: {nan_rows}/{total} ({nan_rows/total:.1%})")

    # Breakdown by reason (only for rows where dx/dy invalid)
    breakdown = (out.loc[out["dx_new"].isna() | out["dy_new"].isna(), "dxdy_reason"]
                   .replace("", "unknown")
                   .value_counts(dropna=False))
    print("\nNaN breakdown by reason:")
    print(breakdown.to_string())

    # Optional: show a few problematic examples
    print("\nExamples of issues (up to 10 rows):")
    print(out.loc[(out["dxdy_reason"] != ""), 
                  [sid_col, lm_col, agei_col, x_col, y_col, "x_next", "y_next", "agei_gap", "dxdy_reason"]]
            .head(10)
            .to_string(index=False))

    return out




In [ ]:
# --- USAGE ---
df = pd.read_csv("/data/all_landmark_series_long.csv")
df2 = recompute_dxdy_with_diagnostics(df)
# If you want to keep the recomputed dx/dy, assign back:
df["dx"] = df2["dx_new"]
df["dy"] = df2["dy_new"]

[dx,dy] rows with NaN: 34033/67269 (50.6%)

NaN breakdown by reason:
dxdy_reason
missing_xy    34033

Examples of issues (up to 10 rows):
sid  landmark  age_idx     x      y  x_next  y_next  agei_gap dxdy_reason
001       ans       10  78.8  -53.9     NaN     NaN       NaN  missing_xy
001 articular       10 -21.5  -36.8     NaN     NaN       NaN  missing_xy
001    basion       10 -31.6  -40.8     NaN     NaN       NaN  missing_xy
001   condyle       10 -19.9  -22.9     NaN     NaN       NaN  missing_xy
001  gonion l       10  -3.1 -101.5     NaN     NaN       NaN  missing_xy
001  gonion u       10  -2.9  -95.6     NaN     NaN       NaN  missing_xy
001  l 6 apex        8  27.0  -98.0     NaN     NaN       1.0  missing_xy
001  l 6 apex        9   NaN    NaN     NaN     NaN       1.0  missing_xy
001  l 6 apex       10   NaN    NaN     NaN     NaN       NaN  missing_xy
001  l 6 cusp        8  38.6  -79.1     NaN     NaN       1.0  missing_xy


C:\Users\DELL\AppData\Local\Temp\ipykernel_9488\4140524686.py:69: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = out.groupby([sid_col, lm_col], group_keys=False).apply(_per_group)


In [20]:


# Replace old dx/dy with new ones
df = (
    df2
    .drop(columns=["dx", "dy"], errors="ignore")     # drop any existing dx/dy
    .rename(columns={"dx_new":"dx", "dy_new":"dy"})  # promote new columns
)

# (Optional) drop diagnostics if you don't want them in the final frame
df = df.drop(columns=["x_next","y_next","agei_next","agei_gap","dxdy_reason","dup_age_idx"], errors="ignore")

# (Optional) sanity check
assert {"dx","dy"}.issubset(df.columns), "dx/dy missing after rename!"
print("Replaced dx, dy with recomputed values.")


Replaced dx, dy with recomputed values.


In [ ]:
df.to_csv("/data/all_landmark_series_long_tcnn.csv", index=False)

In [22]:
df[['sid','landmark', 'age_idx','age','x','y','dx','dy']]

,sid,landmark,age_idx,age,x,y,dx,dy
0,001,ans,1,6.92,67.5,-44.4,1.3,-0.8
1,001,ans,2,7.75,68.8,-45.2,1.6,-1.2
2,001,ans,3,8.50,70.4,-46.4,0.2,-1.9
3,001,ans,4,9.58,70.6,-48.3,0.3,-1.6
4,001,ans,5,10.67,70.9,-49.9,2.7,-1.0
...,...,...,...,...,...,...,...,...
67264,U98,u_6_mcp,10,14.00,35.0,-67.7,NaN,NaN
67265,U98,u_6_mcp,11,15.50,NaN,NaN,NaN,NaN
67266,U98,u_6_mcp,12,17.00,NaN,NaN,NaN,NaN
67267,U98,u_6_mcp,13,19.92,NaN,NaN,NaN,NaN


In [13]:
df2[['x', 'y', "dx_new", "dy_new"]] = df[['x', 'y', "dx", "dy"]].replace([np.inf, -np.inf], np.nan)

bad = df2[['x', 'y',]].isna().any(axis=1)
if bad.any():
    print(f"Dropping {bad.sum()} rows with NaN/inf in x/y.")
    # df = df[~bad].copy()

Dropping 21661 rows with NaN/inf in x/y.


In [ ]:
import pandas as pd
from pathlib import Path

# --------------------
# Inputs (edit paths)
# --------------------
TCN_PATH   = Path("/data/all_landmark_series_long_tcnn.csv")     # the file shown in your screenshot
LONG_FILES = [
    Path("../TCN/B0277_long.csv"),
    Path("../TCN/U69_long.csv"),
    # add more *_long.csv here if needed
]
OUT_PATH   = Path("tcn_dataset_merged.csv")

# --------------------
# Load data
# --------------------
tcn = pd.read_csv(TCN_PATH)
long = pd.concat([pd.read_csv(p) for p in LONG_FILES], ignore_index=True)



In [29]:
long

,sid,sex,angle_class,landmark,age_idx,age,x,y,dx,dy
0,B0277,M,Class III,SELLA,1,12.25,0.0,0.0,0.0,0.0
1,B0277,M,Class III,SELLA,2,13.25,0.0,0.0,0.0,0.0
2,B0277,M,Class III,SELLA,3,14.33,0.0,0.0,0.0,0.0
3,B0277,M,Class III,SELLA,4,15.33,0.0,0.0,0.0,0.0
4,B0277,M,Class III,SELLA,5,16.33,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
307,U69,F,Class II,ARTICULAR,2,10.25,-21.6,-27.7,-2.4,-1.7
308,U69,F,Class II,ARTICULAR,3,11.17,-22.3,-28.1,-4.7,-2.8
309,U69,F,Class II,ARTICULAR,4,13.00,-24.6,-29.3,-4.8,-3.3
310,U69,F,Class II,ARTICULAR,5,14.08,-24.6,-29.7,-4.0,-3.1


In [ ]:
tcn_filtered = tcn[~tcn["sid"].isin(["B0277", "U69"])]

In [38]:
full = pd.concat([tcn_filtered,long],ignore_index=True)

In [ ]:
full.to_csv("/data/all_landmark_series_long_tcnn_full.csv",index=False)

In [ ]:
df = pd.read_csv("/data/all_landmark_series_long_tcnn_full.csv")
for c in ["dx", "dy"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").round(4)
df.to_csv("/data/all_landmark_series_long_tcnn.csv", index=False)